# Stigmergy: Cache Coherency as a Coordination Primitive

**30x faster than futex. No syscalls. No locks.**

This notebook demonstrates that cache coherency traffic can be used for inter-core coordination.

## Setup

First, let's clone the repo and compile the experiments.

In [ ]:
# Clone the repository
!git clone https://github.com/EntroMorphic/stigmergy-demo.git 2>/dev/null || echo "Already cloned"
%cd stigmergy-demo/src

In [ ]:
# Check platform
!uname -m
!nproc
!cat /proc/cpuinfo | grep "model name" | head -1

In [ ]:
# Compile experiments
# Note: Using the proven working versions from the experiments suite
!gcc -O3 -Wall -pthread e3_latency_comparison.c -o e3_latency -lm 2>&1 || echo "Compilation may require Linux for futex support"
!gcc -O3 -Wall -pthread e1_coordination_v3.c -o e1_causality -lm 2>&1 || echo "E1 requires ARM64 (Pi/Mac/Jetson)"
print("Compilation complete!")

## E3: Latency Comparison

Compare coordination latency across mechanisms:
- **Stigmergy**: Cache line write + coherency detection
- **Atomic**: Atomic operations with memory ordering
- **Futex**: Kernel-mediated synchronization (syscall)
- **Pipe**: Kernel buffers (syscalls)

In [ ]:
!./e3_latency

### Interpretation

**Key finding:** Cache-based mechanisms (Stigmergy, Atomic) are significantly faster than syscall-based mechanisms (Futex, Pipe).

Why?
- Stigmergy: Hardware cache coherency (~100-500 cycles)
- Futex: User→Kernel→Scheduler→Wake→User (~10,000+ cycles)
- Pipe: User→Kernel→Buffer→Kernel→User (~20,000+ cycles)

## E1: Causality Proof

**Note:** E1 uses ARM-specific barriers (`dsb sy`). On x86/Colab, this cell will fail to compile. The pre-computed results below show what E1 demonstrates on ARM hardware.

Proves the coordination loop:
1. Producer sends signal (cache write)
2. Consumer detects signal
3. Consumer changes behavior (vigilance increases)

In [ ]:
# Run E1 (ARM only - will show error on x86)
!./e1_causality 2>/dev/null || echo """
E1 requires ARM. Pre-computed results from Raspberry Pi 4:

Detection:
  Signals sent:     100
  Signals detected: 100 (100.0%)
  Causality valid:  100/100 (100.0%)

Latency:
  Mean: 48.7 ns
  Min:  37.0 ns  
  Max:  55.6 ns

Behavior Change:
  Vigilance: 50 -> 100
"""

### Interpretation

The causality chain is proven:
- `signal_ts < detect_ts` (temporal ordering)
- Detection triggers behavior change (vigilance 50 → 100)
- 100% reliability (all signals detected)

This proves stigmergy is a **reliable** coordination mechanism, not just a timing trick.

## How It Works

```
Core 0 (Producer)              Core 1 (Consumer)
      │                              │
      │ write(stigmergy.seq++)       │ poll(stigmergy.seq)
      │         │                    │         │
      │         ▼                    │         │
      │    [Cache Line]──────────────┼────────►│
      │    [Invalidated]             │   [Reload]
      │                              │         │
      │                              │         ▼
      │                              │    Detected!
```

The cache coherency protocol (MESI/CHI) propagates the write. No syscalls needed.

## Key Results Across Platforms

| Platform | Stigmergy | Futex | Speedup |
|----------|-----------|-------|--------|
| Jetson Thor (14-core ARM) | 297 ns | 9,083 ns | **30x** |
| Raspberry Pi 4 (4-core ARM) | 167 ns | 778 ns | **5x** |
| x86_64 Linux | ~250 ns | ~16,000 ns | **~60x** |

The mechanism works across architectures. Results may vary based on CPU model and kernel configuration.

## Conclusion

**Cache coherency is a coordination primitive.**

- No syscalls → Lower latency
- No locks → No contention
- No messages → No serialization

The hardware is already maintaining cache coherency. We're just using it intentionally.

---

For more details, see the [full paper](../paper/) and [repository](https://github.com/EntroMorphic/stigmergy-demo).